# Combined Assignment — N-1 Secure Capacity on the Svedala Grid

**You are a planning engineer at the Swedish TSO.** The market operator just
submitted tomorrow's morning-peak dispatch. By 09:00 you must deliver:

1. Verification that the base dispatch is feasible.
2. **Pre-N-1 TTC** for the `ZON_NORR` → `ZON_MITT` corridor.
3. **N-1 secure TTC** — what becomes NTC.
4. Identification of the **binding contingency** at the N-1 limit.

Read `ASSIGNMENT.md` for the brief, deliverables, and grading rubric.

> The helpers in `tools.py` cover everything you have already seen in the
> tutorial notebooks. The new combined logic — **N-1 inside the capacity sweep**,
> and bisection on top — is what you write here.

## 0. Setup — load your personalised snapshot

**Each student in the class receives a different snapshot of the Svedala grid.**
The snapshots vary by scenario (wet hydro, cold winter, dry summer) and by
randomised dispatch within each scenario. Your TTC numbers, your binding
contingency, and your discussion will all therefore differ from your peers'.

> ⚠️ **Academic integrity.** Because every student has a different snapshot,
> copying numbers from a peer will give *wrong* answers, not just suspicious ones.
> Compare *methods* with classmates, never values.

**Steps:**
1. Find your snapshot file in the course platform: `svedala_student_XX.json`,
   where `XX` is the ID you were given.
2. Place it in `student_snapshots/` next to this notebook.
3. Set `STUDENT_ID` below to your two-digit ID.
4. Run the next two cells. **No TODO here** — just verify they run.

In [ ]:
STUDENT_ID = "01"   # ← change this to YOUR student ID (e.g. "07", "23", ...)

import os, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp

from tools import (load_svedala, check_limits, is_feasible,
                   cross_zone_branches, corridor_flow,
                   gsk_pmax, apply_dispatch_shift)

os.makedirs('report', exist_ok=True)

snapshot_path = f'student_snapshots/svedala_student_{STUDENT_ID}.json'
if not os.path.isfile(snapshot_path):
    raise FileNotFoundError(
        f'Snapshot {snapshot_path!r} not found. '
        f'Download svedala_student_{STUDENT_ID}.json from the course platform '
        f'and place it under student_snapshots/.')

print(f'Using snapshot: {snapshot_path}')

## 1. Base case verification  ★

Load the dispatch, run the power flow, decide whether it is feasible.

**Deliverable:** assign `BASE_OK = True/False` and print a one-line summary.

In [ ]:
net = pp.from_json(snapshot_path)

# TODO 1.a  Run the AC power flow with try/except.
converged = ...

# TODO 1.b  Use check_limits() to count violations and decide BASE_OK.
BASE_OK = ...

# TODO 1.c  Print a one-line summary like:
#   'Base case: converged=True  feasible=True  (0 line, 0 trafo, 0 V_low, 0 V_high)'
...

> ⚠️ **Stop if `BASE_OK` is False.** A capacity number on top of an infeasible
> base case is meaningless.

Note: a CIM-imported real-world model may have small base-case violations even
after a sensible dispatch. If your `check_limits` returns a few violations,
investigate them in your discussion (Task 7) but proceed with the rest of the
tasks — the gap between pre-N-1 and N-1-secure TTC is still meaningful.

## 2. Define the corridor & compute pre-N-1 TTC  ★★

Set up the NORR → MITT corridor and compute the pre-N-1 TTC with
coarse-then-bisect.

**Deliverable:** the values `TTC_pre_n1` and `TTC_pre_n1_limiting`.

In [ ]:
ZONE_SOURCE = 'ZON_NORR'
ZONE_SINK   = 'ZON_MITT'

# Only check the HV transmission grid. The simplified Svedala model lacks
# reactive compensation at 135 kV and below, so those buses violate voltage
# limits even in the base case. 220 kV and 400 kV are the study system.
MIN_KV = 220.0

ZONE_A_BUSES = net.bus[net.bus.SubGeographicalRegion_name == ZONE_SOURCE].index.tolist()
ZONE_B_BUSES = net.bus[net.bus.SubGeographicalRegion_name == ZONE_SINK  ].index.tolist()

LINE_DIR, TRAFO_DIR = cross_zone_branches(net, ZONE_A_BUSES, ZONE_B_BUSES)
GSK_A = gsk_pmax(net, ZONE_A_BUSES)
GSK_B = gsk_pmax(net, ZONE_B_BUSES)
P0 = corridor_flow(net, LINE_DIR, TRAFO_DIR)

print(f'Base corridor flow {ZONE_SOURCE} → {ZONE_SINK}: {P0:+.2f} MW')
print(f'Corridor: {len(LINE_DIR)} line(s), {len(TRAFO_DIR)} trafo(s)')


In [ ]:
# TODO 2.a  Coarse pre-N-1 sweep. At each step:
#   - copy net, apply_dispatch_shift, runpp
#   - record (delta, corridor_flow, max_loading, min/max V, feasible)
# Stop at the first infeasible step.
# (Refer to capacity course notebook 5.)

def coarse_pre_n1_sweep(base_net, step_mw=50.0, max_delta_mw=2500.0):
    rows = []
    delta = 0.0
    while delta <= max_delta_mw:
        # TODO: build one row and append it
        # TODO: break out of the loop on infeasible / non-converged
        delta += step_mw
    return pd.DataFrame(rows)

sweep_pre = coarse_pre_n1_sweep(net)
sweep_pre.tail()

In [ ]:
# TODO 2.b  Bisect to refine TTC_pre_n1 to <= 1 MW resolution.
TTC_pre_n1 = ...
TTC_pre_n1_limiting = ...

print(f'Pre-N-1 TTC NORR → MITT: {TTC_pre_n1:+.2f} MW  '
      f'(limiting: {TTC_pre_n1_limiting})')

## 3. N-1 verification of the base case  ★★

Run an N-1 screen on the base dispatch (no transfer increase yet) and list any
contingencies that already violate.

**Deliverable:** `BASE_CRITICAL_CONTINGENCIES` (may be empty).

In [ ]:
# TODO 3.a  Build the contingency list (lines + trafos in service).
contingencies = ...   # list of (kind, idx) tuples
print(f'{len(contingencies)} credible single contingencies on Svedala')

In [ ]:
# TODO 3.b  Loop over contingencies on a fresh copy of the net,
# trip each, run PF, record feasibility, restore.
# Use try/finally. Refer to N-1 course notebook 5.

def base_n1_screen(base_net, contingencies):
    rows = []
    for kind, idx in contingencies:
        # TODO: implement
        pass
    return pd.DataFrame(rows)

n1_base = base_n1_screen(net, contingencies)
BASE_CRITICAL_CONTINGENCIES = n1_base[~n1_base.feasible] if 'feasible' in n1_base.columns else n1_base.iloc[:0]
print(f'{len(BASE_CRITICAL_CONTINGENCIES)} contingencies are already critical in the base case')

> If the base case is already N-1 critical (some contingency violates *before*
> any extra transfer), the N-1 secure TTC is **negative** — the grid is
> already over its secure limit. Make sure your code handles this gracefully.

## 4. ⭐ N-1 secure capacity sweep  ★★★★ (CORE)

**This is the central deliverable.** Combine the contingency loop from Task 3
with the transfer sweep from Task 2:

1. For each requested ΔP (step by `step_mw`):
   - apply the dispatch on a fresh copy → `net_d`
   - run base PF on `net_d`; if infeasible → step is not N-1 secure, stop
   - for every contingency: deep-copy `net_d`, trip the element, run PF, check
     feasibility
   - the step is **N-1 secure** iff base is feasible *and* every contingency
     is feasible
2. Stop at the first non-N-1-secure step.

Track the *binding* contingency at the limit.

**Deliverable:** the function `n1_secure_sweep` and the DataFrame `sweep_n1`.

In [ ]:
def n1_secure_sweep(base_net, gsk_a, gsk_b, line_dir, trafo_dir,
                    contingencies,
                    step_mw=50.0, max_delta_mw=2500.0,
                    v_min=0.95, v_max=1.05,
                    line_limit=100.0, trafo_limit=100.0,
                    verbose=False):
    """
    Returns a DataFrame with one row per delta step:
      - delta_mw, corridor_flow,
      - base_feasible (bool),
      - n_critical_contingencies (int),
      - first_critical_kind, first_critical_idx (str / int) or None,
      - n_minus_1_secure (bool).
    """
    rows = []
    delta = 0.0
    while delta <= max_delta_mw:
        # ---------------------------------------------------------------
        # TODO 4.a  Apply the shift on a fresh copy and run the base PF.
        # TODO 4.b  If the base case is infeasible, record the row with
        #           n_minus_1_secure = False and BREAK.
        # TODO 4.c  Else loop over contingencies: deep-copy the post-shift
        #           net, trip the element, run PF, check feasibility.
        #           Track the FIRST critical contingency you find.
        # TODO 4.d  Append a row with the documented fields.
        # TODO 4.e  If this step is not N-1 secure, BREAK.
        # ---------------------------------------------------------------
        delta += step_mw
    return pd.DataFrame(rows)

sweep_n1 = n1_secure_sweep(net, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR,
                           contingencies, step_mw=100.0, max_delta_mw=2500.0,
                           verbose=True)
sweep_n1

In [ ]:
# Schema check
expected_cols = {'delta_mw', 'corridor_flow', 'base_feasible',
                 'n_critical_contingencies', 'first_critical_kind',
                 'first_critical_idx', 'n_minus_1_secure'}
assert expected_cols.issubset(set(sweep_n1.columns)), \
    f'Missing columns: {expected_cols - set(sweep_n1.columns)}'
print('Schema OK')

## 5. Bisection on N-1 secure TTC  ★★★

Refine the coarse N-1 secure TTC to ±1 MW. Reuse Task 4's inner check.

**Deliverable:** `bisect_n1_ttc` and `TTC_n1`.

In [ ]:
def n1_secure_at_delta(base_net, delta_mw, gsk_a, gsk_b, contingencies, **kw):
    """True iff `delta_mw` is N-1 secure. Also returns the limiting element."""
    # TODO 5.a  Apply dispatch, run base PF, then loop contingencies.
    # Return (is_secure: bool, limiting_kind: str|None, limiting_idx: int|None).
    ...

def bisect_n1_ttc(base_net, gsk_a, gsk_b, line_dir, trafo_dir, contingencies,
                  lo_delta, hi_delta, tol_mw=1.0, max_iter=15, **kw):
    # TODO 5.b  Standard bisection. Track the last secure midpoint and
    # the limiting element observed at the boundary.
    # Return dict: {'ttc_delta_mw', 'ttc_flow_mw', 'limiting_kind', 'limiting_idx'}.
    ...

secure_steps   = sweep_n1[sweep_n1.n_minus_1_secure]
insecure_steps = sweep_n1[~sweep_n1.n_minus_1_secure]
lo = float(secure_steps.delta_mw.iloc[-1])  if len(secure_steps)   else 0.0
hi = float(insecure_steps.delta_mw.iloc[0]) if len(insecure_steps) else lo + 100.0

ttc_n1 = bisect_n1_ttc(net, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR, contingencies, lo, hi)
TTC_n1 = ttc_n1['ttc_flow_mw']
TTC_n1_limiting = f"{ttc_n1['limiting_kind']} #{ttc_n1['limiting_idx']}"
print(f'N-1 secure TTC NORR → MITT: {TTC_n1:+.2f} MW  (limiting: {TTC_n1_limiting})')

## 6. Capacity report  ★★

Summary table, comparison plot, **map of the binding contingency on the**
**Svedala topology**.

**Deliverable:** files in `report/` plus the comparison plot.

In [ ]:
# TODO 6.a  Build a small comparison DataFrame.
capacity_table = pd.DataFrame([
    {'metric': 'Base flow',         'value_mw': round(P0, 1)},
    {'metric': 'Pre-N-1 TTC',       'value_mw': round(TTC_pre_n1, 1),
     'limiting': TTC_pre_n1_limiting},
    {'metric': 'N-1 secure TTC',    'value_mw': round(TTC_n1, 1),
     'limiting': TTC_n1_limiting},
    {'metric': 'Loss of capacity due to N-1',
     'value_mw': round(TTC_pre_n1 - TTC_n1, 1)},
])
capacity_table

In [ ]:
# TODO 6.b  Save report files: capacity_table.csv, summary.json, report.md
...

In [ ]:
# TODO 6.c  Plot pre-N-1 trajectory and overlay both TTCs as vertical lines.
fig, ax = plt.subplots(figsize=(10, 5))
# ax.plot(sweep_pre.corridor_flow, sweep_pre.max_line_loading, 'o-')
# ax.axvline(TTC_pre_n1, color='tab:orange', linestyle='--', label=f'TTC = {TTC_pre_n1:.0f} MW')
# ax.axvline(TTC_n1,     color='tab:red',    linestyle='--', label=f'N-1 TTC = {TTC_n1:.0f} MW')
# ax.axhline(100, color='red', linestyle=':')
ax.set_xlabel('Corridor flow NORR → MITT [MW]')
ax.set_ylabel('Max line loading [%]')
ax.set_title('Capacity headroom — pre-N-1 vs N-1 secure')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

In [ ]:
# TODO 6.d  (Bonus, +5%)
# Map overlay: plot Svedala by zone with the binding contingency highlighted
# in red. Use net.bus.geo for coordinates - see capacity course notebook 6.
...

## 7. Discussion  ★★

Replace the bullets with **your own** answers, referring to your numbers.
Three to six sentences per question.

1. **By how much, in MW and in %, is N-1 secure TTC below pre-N-1 TTC?** Is
   this in the typical range (10-25 %)?
2. **Which contingency binds the N-1 secure TTC?** Where is it on the map?
   Why is it the binding one (line vs trafo, voltage level, geography)?
3. **What corrective action would you propose** to recover part of the lost
   capacity? Be concrete (re-dispatch generator G by X MW, re-rate line L,
   install a series compensator, ...).
4. **How robust is your N-1 secure TTC to GSK choice?** Pmax-proportional vs
   headroom-proportional vs uniform — would they materially differ?
5. **What assumption** in this study should an operator be warned about
   before publishing the result as NTC? (Hint: think about the ≥80
   contingencies vs reality, the missing TRM, the slack location, ...)

---

### Your answers

*1. ...*

*2. ...*

*3. ...*

*4. ...*

*5. ...*

## Submission checklist

- [ ] Every code cell runs end-to-end (`Restart Kernel & Run All`).
- [ ] `BASE_OK`, `TTC_pre_n1`, `TTC_n1`, and `TTC_n1_limiting` have sensible values.
- [ ] The schema check after Task 4 passes.
- [ ] `report/` contains `capacity_table.csv`, `summary.json`, `report.md`.
- [ ] Discussion answers are written in your own words.
- [ ] Notebook renamed `assignment_<your_name>.ipynb`.

Zip the notebook + `report/` folder and submit on the course platform.